### Hüpfburg – Lösung Schritt für Schritt

*41. Bundeswettbewerb Informatik (2022), 1. Runde, Aufgabe 5*

Sasha und Mika durchlaufen miteinander nach folgenden Regeln einen mit Kreide auf den Schulhof gemalten Parcours:

1. Sasha startet auf Feld 1 und Mika auf Feld 2.
2. In jedem Schritt springen beide gleichzeitig, jeweils entlang eines Pfeils, auf ein neues Feld.
3. Der Parcours gilt als erfolgreich absolviert, sobald beide auf demselben Feld landen.

Den Beispiel-Parcours (`huepfburg0.txt`, 20 Felder) können sie in drei Schritten erfolgreich absolvieren, indem sie gleichzeitig
auf Feld 10 landen. Es gibt aber auch Parcours, für die das nicht möglich ist.

#### Aufgabe 5

Überlege dir, wie man feststellen kann, ob Sasha und Mika einen Parcours erfolgreich absolvieren können. Schreibe ein Programm,
das einen Parcours einliest und dann ausgibt, ob dieser Parcours erfolgreich absolviert werden kann. Falls es möglich ist, soll
außerdem eine erfolgreiche Folge von Sprüngen für Sasha und Mika ausgegeben werden.

Wende dein Programm mindestens auf alle Beispiele an, die du auf den BWINF-Webseiten findest, und dokumentiere die Ergebnisse.

#### Beispieldaten (`beispieldaten/huepfburg0.txt` bis `huepfburg4.txt`)

Erste Zeile: Anzahl der Felder und Anzahl der Pfeile (in `huepfburg0.txt`: `20 44`); danach je Zeile ein Pfeil `von nach`.

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Graphen:** Felder sind Knoten, Pfeile sind (gerichtete) Kanten. Man speichert sie als Nachbarliste: Wörterbuch von Feld zu Liste der Felder,
  auf die man von dort springen kann.
- **Zwei Figuren zu einer machen:** Statt Sasha und Mika einzeln zu verfolgen, betrachtet man das *Paar* `(Feld von Sasha, Feld von Mika)`
  als einen einzigen Zustand. Ein Schritt des Paars ist ein Sprung von beiden zugleich.
- **Breitensuche (BFS):** Findet in einem Graphen ohne Gewichte den Weg mit den wenigsten Schritten und zeigt auch, wenn es gar keinen gibt.
- **Vorgänger merken:** Zu jedem besuchten Zustand merkt man sich, woher man gekommen ist. Rückwärts vom Treffer bis zum Start
  ergibt das den Weg.

**Nützliche Funktionen und Bibliotheken**
- `collections.deque` mit `append` und `popleft()` als Warteschlange (eine Liste mit `pop(0)` ginge auch, ist aber langsamer).
- Tupel als Schlüssel in einem Wörterbuch, z. B. `vorgaenger[(1, 2)]`.

## Die Idee

Es gibt zwei Figuren, die sich gleichzeitig bewegen. Das lässt sich nicht direkt mit einer normalen Suche im Graphen lösen. Der Trick:
Wir betrachten ein **Paar** `(s, m)`, mit `s` = Feld von Sasha und `m` = Feld von Mika. Bei `n` Feldern gibt es nur `n * n` solche Paare.

- Start ist das Paar `(1, 2)`.
- Von `(s, m)` kommt man zu `(s2, m2)`, wenn es einen Pfeil `s -> s2` **und** einen Pfeil `m -> m2` gibt. Es gibt also für jedes Paar
  von Pfeilen, die bei `s` und bei `m` beginnen, einen Nachfolger.
- Ziel ist jedes Paar `(f, f)` mit zwei gleichen Feldern.

Gesucht ist also ein Weg im Paar-Graphen vom Start zu irgendeinem Ziel. Die Breitensuche besucht jedes Paar höchstens einmal und findet
einen kürzesten Weg. Hat sie die Warteschlange leergeräumt, ohne ein Ziel zu sehen, gibt es sicher keine Lösung (die Abbruchbedingung
ist damit automatisch begründet: Es gibt nur endlich viele Paare).

Wichtig: Niemand darf „warten“. Hat ein Feld keinen Pfeil, von dem man weiterspringen könnte, hat das Paar dort einfach keinen Nachfolger.

## Schritt 1: Einlesen

Wir lesen alle Zahlen der Datei und bauen die Nachbarlisten: Jedes Feld `1` bis `n` bekommt eine Liste der Felder, auf die man von ihm aus springen kann.

In [ ]:
def lies_parcours(dateiname):
    with open(dateiname, encoding='utf-8-sig') as f:
        zahlen = [int(z) for z in f.read().split()]
    n, anzahl_pfeile = zahlen[0], zahlen[1]
    nachbarn = {feld: [] for feld in range(1, n + 1)}
    for i in range(anzahl_pfeile):
        von, nach = zahlen[2 + 2 * i], zahlen[3 + 2 * i]
        nachbarn[von].append(nach)
    return nachbarn


parcours = lies_parcours('beispieldaten/huepfburg0.txt')
print(len(parcours), 'Felder')
print('Von Feld 1 aus:', parcours[1])
print('Von Feld 2 aus:', parcours[2])

20 Felder
Von Feld 1 aus: [18, 8, 4]
Von Feld 2 aus: [3, 19]


## Schritt 2: Breitensuche über die Paare

`vorgaenger` ist zugleich die Liste der besuchten Paare: Ein Paar steht genau dann darin, wenn wir es schon gesehen haben. Für den Start
steht `None` (kein Vorgänger).

In der Schleife nehmen wir das vorderste Paar aus der Warteschlange. Ist es ein Ziel (`s == m`), sind wir fertig. Sonst hängen wir alle
neuen Nachfolger hinten an.

Da die Breitensuche Paare in der Reihenfolge ihrer Entfernung vom Start abarbeitet, ist der erste gefundene Treffer ein kürzester.

In [ ]:
from collections import deque


def suche_treffer(nachbarn, start=(1, 2)):
    vorgaenger = {start: None}
    warteschlange = deque([start])
    while warteschlange:
        paar = warteschlange.popleft()
        s, m = paar
        if s == m:
            return paar, vorgaenger
        for s2 in nachbarn[s]:
            for m2 in nachbarn[m]:
                neu = (s2, m2)
                if neu not in vorgaenger:
                    vorgaenger[neu] = paar
                    warteschlange.append(neu)
    return None, vorgaenger


treffer, vorgaenger = suche_treffer(parcours)
print('Treffer:', treffer)
print('besuchte Paare:', len(vorgaenger), 'von', 20 * 20)

Treffer: (10, 10)
besuchte Paare: 63 von 400


## Schritt 3: Den Weg zurückverfolgen

Vom Treffer aus gehen wir über die gemerkten Vorgänger bis zum Start zurück und sammeln dabei in zwei Listen die Felder von Sasha und von Mika.
Weil wir von hinten sammeln, drehen wir die Listen am Ende um.

In [ ]:
def weg_aus_vorgaengern(treffer, vorgaenger):
    sasha, mika = [], []
    paar = treffer
    while paar is not None:
        sasha.append(paar[0])
        mika.append(paar[1])
        paar = vorgaenger[paar]
    sasha.reverse()
    mika.reverse()
    return sasha, mika


sasha, mika = weg_aus_vorgaengern(treffer, vorgaenger)
print('Sasha:', sasha)
print('Mika: ', mika)

Sasha: [1, 18, 13, 10]
Mika:  [2, 19, 20, 10]


## Schritt 4: Alles zusammen und ein Kontrollprogramm

`loese(dateiname)` liest ein, sucht und gibt das Ergebnis aus. Lange Wege (bei `huepfburg1` 121 Sprünge) kürzen wir bei der Ausgabe.

Weil die Aufgabe verlangt, dass die Sprungfolge **korrekt** ist, gibt es dazu `pruefe_weg`: Es kontrolliert, dass beide auf Feld 1 bzw. 2 starten,
dass jeder Sprung wirklich entlang eines Pfeils geht, dass beide gleich viele Sprünge machen und dass sie am Ende auf demselben Feld stehen.

In [ ]:
def pruefe_weg(nachbarn, sasha, mika):
    ok = sasha[0] == 1 and mika[0] == 2 and len(sasha) == len(mika)
    for i in range(1, len(sasha)):
        ok = ok and sasha[i] in nachbarn[sasha[i - 1]] and mika[i] in nachbarn[mika[i - 1]]
    return ok and sasha[-1] == mika[-1]


def kurz(weg):
    text = ' -> '.join(str(f) for f in weg[:12])
    return text + ' -> ...' if len(weg) > 12 else text


def loese(dateiname):
    nachbarn = lies_parcours(dateiname)
    treffer, vorgaenger = suche_treffer(nachbarn)
    if treffer is None:
        print('Keine Lösung:', len(vorgaenger), 'von', len(nachbarn) ** 2, 'Paaren erreichbar, kein Paar mit gleichem Feld.')
        return None
    sasha, mika = weg_aus_vorgaengern(treffer, vorgaenger)
    assert pruefe_weg(nachbarn, sasha, mika)
    print('Lösung mit', len(sasha) - 1, 'Sprüngen, beide landen auf Feld', treffer[0])
    print('Sasha:', kurz(sasha))
    print('Mika: ', kurz(mika))
    return len(sasha) - 1


for nr in range(5):
    print('huepfburg' + str(nr) + '.txt')
    loese('beispieldaten/huepfburg' + str(nr) + '.txt')
    print()

huepfburg0.txt
Lösung mit 3 Sprüngen, beide landen auf Feld 10
Sasha: 1 -> 18 -> 13 -> 10
Mika:  2 -> 19 -> 20 -> 10

huepfburg1.txt
Lösung mit 121 Sprüngen, beide landen auf Feld 4
Sasha: 1 -> 4 -> 5 -> 6 -> 7 -> 8 -> 9 -> 10 -> 11 -> 12 -> 13 -> 14 -> ...
Mika:  2 -> 3 -> 4 -> 5 -> 6 -> 7 -> 8 -> 9 -> 10 -> 11 -> 12 -> 13 -> ...

huepfburg2.txt
Lösung mit 8 Sprüngen, beide landen auf Feld 27
Sasha: 1 -> 51 -> 76 -> 59 -> 42 -> 65 -> 54 -> 92 -> 27
Mika:  2 -> 24 -> 53 -> 2 -> 106 -> 136 -> 108 -> 100 -> 27

huepfburg3.txt
Keine Lösung: 72 von 576 Paaren erreichbar, kein Paar mit gleichem Feld.

huepfburg4.txt
Lösung mit 16 Sprüngen, beide landen auf Feld 12
Sasha: 1 -> 99 -> 89 -> 79 -> 78 -> 77 -> 76 -> 66 -> 56 -> 55 -> 54 -> 44 -> ...
Mika:  2 -> 12 -> 11 -> 100 -> 2 -> 12 -> 11 -> 100 -> 2 -> 12 -> 11 -> 100 -> ...



## Auswertung

- `huepfburg0`: 3 Sprünge, beide landen auf Feld 10 – wie auf dem Aufgabenblatt.
- `huepfburg1`: Lösung gibt es, aber erst nach 121 Sprüngen. Felder werden dabei vielfach wiederholt: Beide laufen lange im Kreis.
- `huepfburg2`: 8 Sprünge. Ein Feld darf mehrfach besucht werden.
- `huepfburg3`: **keine Lösung**. Von den 576 Paaren sind nur 72 vom Start aus erreichbar, und keines davon hat zwei gleiche Felder.
- `huepfburg4`: 16 Sprünge.

Die Längen stimmen mit den offiziellen Lösungshinweisen überein (3, 121, 8, 16; bei Beispiel 3 keine Lösung). Die Wege selbst können sich
unterscheiden, denn es kann mehrere gleich kurze Wege geben.

**Aufwand:** Es gibt `n * n` Paare. Jede Kombination zweier Pfeile wird höchstens einmal angeschaut, das sind höchstens
`(Anzahl Pfeile)²` Schritte. Das ist für diese Aufgabe sehr schnell.

## Für Neugierige

**Wie lang kann der kürzeste Weg höchstens werden?** Jedes Paar wird auf einem kürzesten Weg höchstens einmal besucht. Also ist ein kürzester Weg
höchstens `n * n - 1` Sprünge lang. Wer ohne Breitensuche einfach „spielt“, bis sich die beiden treffen, muss deshalb nach `n * n` Sprüngen aufhören:
Hat bis dahin kein Treffer stattgefunden, kommt auch später keiner mehr. So eine Abbruchbedingung verlangen auch die Bewertungskriterien.

**Mengen von Feldern statt Paaren:** Man kann auch für jeden Schritt die Menge der Felder merken, auf denen Sasha sein kann, und die, auf denen Mika sein kann.
Gibt es ein gemeinsames Feld, ist eine Lösung da. Aber die Sprungfolge muss man dann mühsam rückwärts rekonstruieren, und man braucht eine
eigene Begründung für den Abbruch. Die Paar-Breitensuche ist deshalb sauberer.

**MiniZinc?** Hier muss nur ein kürzester Weg in einem Graphen gefunden werden. Das erledigt die Breitensuche direkt und schnell. Ein Constraint-Modell
müsste dafür die Weglänge raten und wäre nur umständlicher, daher gibt es in diesem Notebook keinen MiniZinc-Abschnitt.